# RF-3DGS 数据集生成 (Etoile 场景, 多辐射源)

基于 RF-3DGS 教程的数据格式，使用 Sionna 内置 etoile 场景生成 MPC（多径分量）空间频谱数据集。

**输出格式 (COLMAP 兼容):**
- `cameras.txt` — 相机内参 (PINHOLE 模型)
- `images.txt` — 相机外参 (四元数旋转 + 平移)
- `spectrum/` — 频谱图像 (PNG, jet colormap)

**特性:**
- 多个全向天线辐射源 (TX)
- 通过 coverage map 确保接收点不在墙内
- MPC/Ideal Spectrum 方式 (无需波束赋形)

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import cv2
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)
tf.get_logger().setLevel('ERROR')

import sionna
import sionna.rt
from sionna.rt import load_scene, Transmitter, Receiver, PlanarArray,PathSolver
from scipy.spatial.transform import Rotation
import matplotlib.pyplot as plt
import imageio

np.random.seed(42)
tf.random.set_seed(42)

## 1. 参数配置

In [ ]:
# ========== 输出目录 ==========
OUTPUT_DIR = './dataset/etoile_demo'

# ========== 场景参数 ==========
SCENE_FREQ = 3.5e9          # 3.5 GHz (5G 中频段)
RX_HEIGHT = 1.5              # 接收器高度 (m)
TX_HEIGHT = 25.0             # 发射器高度 (m), 楼顶基站

# ========== 图像参数 ==========
IMAGE_WIDTH = 300
IMAGE_HEIGHT = 200
FOV = 90                     # 水平视场角 (度)

# ========== 采样参数 ==========
NUM_RX_SAMPLES = 80          # 最终使用的接收点数量
VIEW_ANGLES = [-np.pi/2, 0, np.pi/2, np.pi]  # 每个位置4个朝向

# ========== 射线追踪参数 ==========
MAX_DEPTH = 3                # 最大反射/散射深度
SCAT_KEEP_PROB = 0.3         # 散射保留概率
NUM_RT_SAMPLES = int(1e6)    # 射线数量

# ========== 频谱渲染参数 ==========
IMAGE_SCALE = 3              # 等距圆柱投影分辨率倍数
KERNEL_SIZE = 3              # 高斯核大小
KERNEL_SIGMA = 3             # 高斯核标准差

# ========== Coverage Map 参数 ==========
CM_CELL_SIZE = 5.0           # 覆盖图网格大小 (m)
CM_NUM_SAMPLES = int(2e6)    # 覆盖图射线数量

## 2. 加载场景 & 配置天线

In [ ]:
scene = load_scene(sionna.rt.scene.etoile)
scene.frequency = SCENE_FREQ

# TX: 单个全向天线 (每个辐射源)
scene.tx_array = PlanarArray(
    num_rows=1, num_cols=1,
    vertical_spacing=0.5, horizontal_spacing=0.5,
    pattern="iso", polarization="V"
)

# RX: 单个全向天线 (MPC spectrum 不需要阵列)
scene.rx_array = PlanarArray(
    num_rows=1, num_cols=1,
    vertical_spacing=0.5, horizontal_spacing=0.5,
    pattern="iso", polarization="V"
)

print(f"场景加载完成. 频率: {scene.frequency/1e9:.1f} GHz")

## 3. 添加多个辐射源 (TX)

在场景中不同位置放置全向天线辐射源，模拟多基站场景。
TX 放置在建筑物高度，确保具有良好的覆盖。

In [ ]:
# 清除已有设备
for name in list(scene.transmitters.keys()):
    scene.remove(name)
for name in list(scene.receivers.keys()):
    scene.remove(name)

# 定义多个辐射源位置 (x, y, z)
# etoile 场景以凯旋门为中心，多条大道向外辐射
# TX 放在不同方位的高处
tx_positions = [
    [60,   40,  TX_HEIGHT],    # 东北区域
    [-50, -70,  TX_HEIGHT],    # 西南区域
    [80,  -30,  TX_HEIGHT],    # 东南区域
]

for i, pos in enumerate(tx_positions):
    # sionna expects a sequence of floats for position (not a numpy ndarray)
    tx = Transmitter(name=f"tx_{i}", position=pos)
    scene.add(tx)
    print(f"  TX_{i}: position = {pos}")

num_tx = len(tx_positions)
print(f"\n共添加 {num_tx} 个辐射源")

# 4. 验证用LoS验证位置有效性

In [ ]:
print("正在通过 LoS/反射 验证生成有效接收位置...")

# 在场景范围内生成候选网格
# etoile 场景大约 ±200m, 街道集中在 ±120m 范围内
GRID_SPACING = 15  # 每 15m 一个候选点
grid_x = np.arange(-120, 121, GRID_SPACING)
grid_y = np.arange(-120, 121, GRID_SPACING)

candidates = []
for x in grid_x:
    for y in grid_y:
        # 添加小随机偏移避免所有点落在同一网格线上
        x_offset = np.random.uniform(-GRID_SPACING * 0.3, GRID_SPACING * 0.3)
        y_offset = np.random.uniform(-GRID_SPACING * 0.3, GRID_SPACING * 0.3)
        candidates.append([float(x + x_offset), float(y + y_offset), RX_HEIGHT])

print(f"候选点: {len(candidates)} 个")

# PathSolver 只创建一次 (避免循环内重复创建)
p_solver = PathSolver()

# 验证: 对每个候选位置计算路径, 有信号说明在室外
valid_rx_locs = []
for i, pos in enumerate(candidates):
    if i % 50 == 0:
        print(f"  验证进度: {i}/{len(candidates)}")
    try:
        scene.remove("rx_validate")
    except Exception:
        pass

    rx = Receiver(name="rx_validate", position=pos)
    scene.add(rx)

    try:
        paths = p_solver(
            scene=scene,
            max_depth=1,
            refraction=True,
            diffraction=False,
            diffuse_reflection=False,
            samples_per_src=int(1e5)
        )
        # 检查是否有任何有效路径
        total_gain = float(np.sum(np.abs(paths.a)))
        if total_gain > 0:
            valid_rx_locs.append(pos)
    except Exception:
        pass  # 跳过计算失败的位置

scene.remove("rx_validate")

print(f"\nLoS 验证通过: {len(valid_rx_locs)} / {len(candidates)} 个有效位置")

# 从有效位置中随机采样
if len(valid_rx_locs) > NUM_RX_SAMPLES:
    indices = np.random.choice(len(valid_rx_locs), NUM_RX_SAMPLES, replace=False)
    rx_locs = [valid_rx_locs[i] for i in indices]
else:
    rx_locs = valid_rx_locs

print(f"最终采样: {len(rx_locs)} 个接收点")

In [ ]:
# 可视化 TX 和 RX 位置
rx_locs_arr = np.array(rx_locs)
tx_locs_arr = np.array(tx_positions)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111)

ax.scatter(rx_locs_arr[:, 0], rx_locs_arr[:, 1],
           c='blue', marker='o', s=15, alpha=0.5, label=f'RX ({len(rx_locs)})')
ax.scatter(tx_locs_arr[:, 0], tx_locs_arr[:, 1],
           c='red', marker='^', s=100, label=f'TX ({num_tx})')

for i, pos in enumerate(tx_positions):
    ax.annotate(f'TX_{i}', (pos[0], pos[1]), fontsize=9, color='red')

ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_title('TX/RX 位置分布')
ax.legend()
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"数据集大小: {len(rx_locs)} 个位置 × {len(VIEW_ANGLES)} 个朝向 = {len(rx_locs)*len(VIEW_ANGLES)} 张频谱图")

## 5. 工具函数

包含 COLMAP 格式 I/O、频谱渲染、坐标变换等工具。

In [ ]:
# ==================== COLMAP 格式相关 ====================

class ColmapCamera:
    """COLMAP 相机内参"""
    def __init__(self, id, model, width, height, params):
        self.id = id
        self.model = model
        self.width = width
        self.height = height
        self.params = params   # [fx, fy, cx, cy]

class ColmapImage:
    """COLMAP 图像外参 (world-to-camera)"""
    def __init__(self, id, qvec, tvec, camera_id, name):
        self.id = id
        self.qvec = qvec      # [qw, qx, qy, qz] world-to-camera rotation
        self.tvec = tvec       # world-to-camera translation
        self.camera_id = camera_id
        self.name = name

def save_cameras_txt(path, cameras):
    """保存 cameras.txt (COLMAP 格式)"""
    with open(path, "w") as f:
        f.write("# Camera list with one line of data per camera:\n")
        f.write("# CAMERA_ID, MODEL, WIDTH, HEIGHT, PARAMS[fx,fy,cx,cy]\n")
        for cam_id, cam in cameras.items():
            params_str = " ".join(map(str, cam.params))
            f.write(f"{cam_id} {cam.model} {cam.width} {cam.height} {params_str}\n")

def save_images_txt(path, images):
    """保存 images.txt (COLMAP 格式, w2c 四元数 + w2c 平移)"""
    with open(path, "w") as f:
        f.write("# Image list with two lines of data per image:\n")
        f.write("# IMAGE_ID, QW, QX, QY, QZ, TX, TY, TZ, CAMERA_ID, NAME\n")
        f.write("# POINTS2D[] as (X, Y, POINT3D_ID)\n")
        for img_id, img in images.items():
            qvec_str = " ".join(map(str, img.qvec))
            tvec_str = " ".join(map(str, img.tvec))
            f.write(f"{img_id} {qvec_str} {tvec_str} {img.camera_id} {img.name}\n")
            f.write("\n")  # POINTS2D 空行

def save_points3D_txt(path):
    """保存空的 points3D.txt (COLMAP 兼容性所需)"""
    with open(path, "w") as f:
        f.write("# 3D point list with one line of data per point:\n")
        f.write("# POINT3D_ID, X, Y, Z, R, G, B, ERROR, TRACK[] as (IMAGE_ID, POINT2D_IDX)\n")
        f.write("# Number of points: 0\n")

def calculate_camera_intrinsics(width, height, focal_length):
    """计算 PINHOLE 相机内参"""
    fx = fy = focal_length
    cx = width / 2.0
    cy = height / 2.0
    return fx, fy, cx, cy

# ==================== 坐标变换 ====================

def euler_to_quaternion(euler_angles):
    """
    将 Sionna 的欧拉角 (yaw, pitch, roll) 转换为 COLMAP 格式的
    world-to-camera (w2c) 四元数和旋转矩阵。

    COLMAP images.txt 存储的是 w2c 变换:
      X_cam = R_w2c * X_world + t_w2c
      camera_center = -R_w2c^T * t_w2c

    步骤:
    1. 从 COLMAP 默认相机方向 (+z forward, -y up) 旋转到 Sionna 阵列默认方向 (+x forward, +z up)
    2. 应用 Sionna 阵列的旋转 (ZYX 内旋)
    3. 得到 world-to-camera 旋转 R_w2c
    4. 转换为 scalar-first 四元数 (qw, qx, qy, qz)

    返回: (R_w2c, qvec_w2c)
    """
    # COLMAP camera default → Sionna array default
    R_posz2posx = Rotation.from_euler('ZYX', [-np.pi/2, 0.0, -np.pi/2])

    # Sionna array default → 当前采样方向
    yaw, pitch, roll = euler_angles
    R_posx2array = Rotation.from_euler('ZYX', [yaw, pitch, roll])

    # world-to-camera (W2C): 先做 R_posz2posx 再做 R_posx2array
    R_w2c = R_posx2array * R_posz2posx

    # 转换为 scalar-first 四元数 [qw, qx, qy, qz]
    q = R_w2c.as_quat()  # scipy 返回 [qx, qy, qz, qw]
    qvec_w2c = [q[3], q[0], q[1], q[2]]

    return R_w2c, qvec_w2c

# ==================== 频谱渲染 ====================

def plot_multi_tx_spectrum(paths, num_tx, image_scale=3, kernel_size=3, kernel_sigma=3):
    """
    渲染多 TX 合并的 MPC 空间频谱 (等距圆柱投影)。

    每个多径分量按其 AoA (theta_r, phi_r) 和振幅绘制为高斯点，
    所有 TX 的贡献叠加到同一张图上。

    返回: [360*scale, 180*scale] 的 dB 域频谱
    """
    img = np.zeros((360 * image_scale, 180 * image_scale))

    sigma_x, sigma_y = kernel_sigma, kernel_sigma
    size_x = int(kernel_size * sigma_x) | 1
    size_y = int(kernel_size * sigma_y) | 1
    x = np.linspace(-size_x // 2, size_x // 2, size_x)
    y = np.linspace(-size_y // 2, size_y // 2, size_y)
    x, y = np.meshgrid(x, y)
    gauss_kernel = np.exp(-(x**2 / (2 * sigma_x**2) + y**2 / (2 * sigma_y**2)))

    for tx_idx in range(num_tx):
        theta = paths.theta_r.numpy()[0, 0, tx_idx, :] * 180 / np.pi
        phi = paths.phi_r.numpy()[0, 0, tx_idx, :] * 180 / np.pi
        amps = np.abs(paths.a.numpy()[0, 0, 0, tx_idx, 0, :, 0])

        for idx, intensity in enumerate(amps):
            if intensity > 0:
                path_dot = gauss_kernel * intensity / np.sum(gauss_kernel)

                phi_idx = int(-phi[idx] + 180) * image_scale
                theta_idx = int(theta[idx]) * image_scale
                xmin = max(0, phi_idx - size_x // 2)
                xmax = min(360 * image_scale, phi_idx + size_x // 2 + 1)
                ymin = max(0, theta_idx - size_y // 2)
                ymax = min(180 * image_scale, theta_idx + size_y // 2 + 1)

                gauss_xmin = max(0, size_x // 2 - phi_idx)
                gauss_xmax = min(size_x, 360 * image_scale - phi_idx + size_x // 2)
                gauss_ymin = max(0, size_y // 2 - theta_idx)
                gauss_ymax = min(size_y, 180 * image_scale - theta_idx + size_y // 2)

                img[xmin:xmax, ymin:ymax] += path_dot[gauss_xmin:gauss_xmax, gauss_ymin:gauss_ymax]

    # 转换到 dB 域
    non_zero_mask = img != 0.0
    zero_mask = img == 0.0
    if np.any(non_zero_mask):
        img[non_zero_mask] = 10 * np.log10(img[non_zero_mask])
        img[zero_mask] = np.min(img[non_zero_mask]) - 10
    else:
        img[:] = -100  # 无信号

    return img

def jet_colormap_convert(array_2d):
    """将 2D 数组归一化并应用 jet colormap -> [H, W, 3] RGB float (0~1)"""
    arr_min, arr_max = np.min(array_2d), np.max(array_2d)
    if arr_max - arr_min < 1e-12:
        normalized = np.zeros_like(array_2d)
    else:
        normalized = (array_2d - arr_min) / (arr_max - arr_min)
    jet = plt.get_cmap('jet')
    colored = jet(normalized)[:, :, :3]  # 去掉 alpha
    return colored

# ==================== 等距圆柱 → 透视投影 ====================

def xyz2lonlat(xyz):
    norm = np.linalg.norm(xyz, axis=-1, keepdims=True)
    xyz_norm = xyz / norm
    lon = np.arctan2(xyz_norm[..., 0:1], xyz_norm[..., 2:])
    lat = np.arcsin(xyz_norm[..., 1:2])
    return np.concatenate([lon, lat], axis=-1)

def lonlat2XY(lonlat, shape):
    X = (lonlat[..., 0:1] / (2 * np.pi) + 0.5) * (shape[1] - 1)
    Y = (lonlat[..., 1:] / (np.pi) + 0.5) * (shape[0] - 1)
    return np.concatenate([X, Y], axis=-1)

def equirectangular_to_perspective(img, fov, theta_deg, phi_deg, height, width):
    """
    将等距圆柱投影图像转换为透视投影 (PINHOLE 相机)。

    Args:
        img: 等距圆柱图像 [H, W, C] (float)
        fov: 水平视场角 (度)
        theta_deg: 水平旋转角 (度)
        phi_deg: 垂直旋转角 (度, 90=地平线)
        height, width: 输出图像尺寸

    Returns:
        perspective: 透视图像 [height, width, C]
    """
    if len(img.shape) == 2:
        img = img[..., np.newaxis]

    f = 0.5 * width / np.tan(0.5 * fov / 180.0 * np.pi)
    cx = (width - 1) / 2.0
    cy = (height - 1) / 2.0
    K = np.array([[f, 0, cx], [0, f, cy], [0, 0, 1]], np.float32)
    K_inv = np.linalg.inv(K)

    x = np.arange(width)
    y = np.arange(height)
    x, y = np.meshgrid(x, y)
    z = np.ones_like(x)
    xyz = np.concatenate([x[..., None], y[..., None], z[..., None]], axis=-1)
    xyz = xyz @ K_inv.T

    y_axis = np.array([0.0, 1.0, 0.0], np.float32)
    x_axis = np.array([1.0, 0.0, 0.0], np.float32)
    R1, _ = cv2.Rodrigues(y_axis * np.radians(theta_deg))
    R2, _ = cv2.Rodrigues(np.dot(R1, x_axis) * np.radians(phi_deg))
    R = R2 @ R1
    xyz = xyz @ R.T
    lonlat = xyz2lonlat(xyz)
    XY = lonlat2XY(lonlat, shape=img.shape).astype(np.float32)

    persp = cv2.remap(img.astype(np.float32), XY[..., 0], XY[..., 1],
                      cv2.INTER_CUBIC, borderMode=cv2.BORDER_WRAP)
    return persp

print("工具函数加载完成")

## 6. 数据集生成

对每个接收位置:
1. 计算来自所有 TX 的多径传播
2. 渲染等距圆柱 MPC 频谱 (合并所有 TX 贡献)
3. 对每个观测朝向，转换为透视投影图像
4. 保存频谱图像 + COLMAP 格式的位姿

In [ ]:
def generate_dataset(rx_locs, tx_positions, scene, output_dir,
                     image_width, image_height, fov, view_angles,
                     max_depth, scat_keep_prob, num_rt_samples,
                     image_scale, kernel_size, kernel_sigma):
    """
    生成 RF-3DGS 兼容数据集。

    目录结构:
        output_dir/
            spectrum/            — 频谱图像 (PNG)
            cameras.txt          — COLMAP 相机内参
            images.txt           — COLMAP 图像外参 (w2c 位姿)
            points3D.txt         — COLMAP 3D 点 (空)
            tx_positions.txt     — 辐射源位置
            rx_positions.txt     — 接收点位置
    """
    spectrum_dir = os.path.join(output_dir, 'spectrum')
    os.makedirs(spectrum_dir, exist_ok=True)

    num_tx = len(tx_positions)
    cameras = {}
    images = {}

    # ----- 相机内参 (所有图像共享) -----
    focal_length = image_width / (2 * np.tan(np.deg2rad(fov) / 2))
    fx, fy, cx, cy = calculate_camera_intrinsics(image_width, image_height, focal_length)
    camera_id = 1
    cameras[camera_id] = ColmapCamera(camera_id, "PINHOLE", image_width, image_height,
                                       [float(fx), float(fy), cx, cy])

    # ----- PathSolver 实例 (复用, 避免重复创建) -----
    p_solver = PathSolver()

    # ----- 全局归一化: 先扫描几个位置确定 spec 范围 -----
    print(">>> 计算全局归一化范围...")
    spec_max = None
    spec_min = None
    sample_indices = [0, len(rx_locs) // 2, len(rx_locs) - 1]

    for si in sample_indices:
        pos = rx_locs[si]
        try:
            scene.remove("rx_norm")
        except:
            pass
        rx = Receiver(name="rx_norm", position=pos)
        scene.add(rx)

        paths = p_solver(
            scene=scene,
            max_depth=max_depth,
            samples_per_src=num_rt_samples,
            diffuse_reflection=True,
            refraction=True,
            diffraction=False,
        )

        spec = plot_multi_tx_spectrum(paths, num_tx, image_scale, kernel_size, kernel_sigma).T
        if spec_max is None or np.max(spec) > spec_max:
            spec_max = np.max(spec)
        if spec_min is None or np.min(spec) < spec_min:
            spec_min = np.min(spec)

    try:
        scene.remove("rx_norm")
    except:
        pass
    print(f"    spec 范围: [{spec_min:.2f}, {spec_max:.2f}]")

    # ----- 遍历每个 RX 位置 -----
    image_counter = 1
    jet_colormap = plt.get_cmap('jet')

    for i, rx_loc in enumerate(rx_locs):
        if i % 20 == 0:
            print(f">>> 生成进度: {i}/{len(rx_locs)}")

        # 计算等距圆柱频谱 (一次计算, 所有 TX 合并)
        try:
            scene.remove("rx_gen")
        except:
            pass
        rx = Receiver(name="rx_gen", position=rx_loc)
        scene.add(rx)

        paths = p_solver(
            scene=scene,
            max_depth=max_depth,
            samples_per_src=num_rt_samples,
            diffuse_reflection=True,
            refraction=True,
            diffraction=False,
        )

        spec_equirect = plot_multi_tx_spectrum(paths, num_tx, image_scale, kernel_size, kernel_sigma).T

        # 归一化到 [0, 1]
        spec_normalized = np.clip((spec_equirect - spec_min) / (spec_max - spec_min + 1e-12), 0, 1)

        # jet colormap -> [H, W, 3] float (0~1)
        jet_image = jet_colormap(spec_normalized)[:, :, :3]

        # 对每个观测朝向生成透视图
        for angle in view_angles:
            orientation = [float(angle), 0.0, 0.0]  # yaw, pitch, roll

            # 计算 COLMAP 位姿 (world-to-camera)
            R_w2c, qvec_w2c = euler_to_quaternion(orientation)
            tvec_w2c = -R_w2c.apply(np.array(rx_loc))

            # 等距圆柱 → 透视投影
            perspective_img = equirectangular_to_perspective(
                jet_image, fov, float(angle * 180 / np.pi), 90,
                image_height, image_width
            )
            perspective_img = np.clip(perspective_img, 0, 1)

            # 保存频谱图像
            image_name = f'{image_counter:05d}.png'
            spectrum_path = os.path.join(spectrum_dir, image_name)
            imageio.imwrite(spectrum_path, (perspective_img * 255).astype(np.uint8))

            # 记录 COLMAP 外参 (w2c)
            images[image_counter] = ColmapImage(
                image_counter, qvec_w2c, tvec_w2c.tolist(),
                camera_id, image_name
            )

            image_counter += 1

    try:
        scene.remove("rx_gen")
    except:
        pass

    # ----- 保存 COLMAP 文件 -----
    save_cameras_txt(os.path.join(output_dir, 'cameras.txt'), cameras)
    save_images_txt(os.path.join(output_dir, 'images.txt'), images)
    save_points3D_txt(os.path.join(output_dir, 'points3D.txt'))

    # ----- 保存辅助元数据 -----
    np.savetxt(os.path.join(output_dir, 'tx_positions.txt'), np.array(tx_positions),
               header='x y z', comments='')
    np.savetxt(os.path.join(output_dir, 'rx_positions.txt'), np.array(rx_locs),
               fmt='%.4f', header='x y z', comments='')

    total_images = image_counter - 1
    print(f"\n{'='*50}")
    print(f"数据集生成完成!")
    print(f"  输出目录: {output_dir}")
    print(f"  频谱图像: {total_images} 张 ({image_width}x{image_height})")
    print(f"  接收位置: {len(rx_locs)} 个")
    print(f"  辐射源:   {num_tx} 个")
    print(f"{'='*50}")

    return total_images

In [ ]:
# 运行数据集生成
total = generate_dataset(
    rx_locs=rx_locs,
    tx_positions=tx_positions,
    scene=scene,
    output_dir=OUTPUT_DIR,
    image_width=IMAGE_WIDTH,
    image_height=IMAGE_HEIGHT,
    fov=FOV,
    view_angles=VIEW_ANGLES,
    max_depth=MAX_DEPTH,
    scat_keep_prob=SCAT_KEEP_PROB,
    num_rt_samples=NUM_RT_SAMPLES,
    image_scale=IMAGE_SCALE,
    kernel_size=KERNEL_SIZE,
    kernel_sigma=KERNEL_SIGMA,
)

## 7. 验证输出

检查生成的数据集格式是否正确。

In [ ]:
# 检查目录结构
print("=== 目录结构 ===")
for root, dirs, files in os.walk(OUTPUT_DIR):
    level = root.replace(OUTPUT_DIR, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = '  ' * (level + 1)
    for f in sorted(files)[:5]:
        filepath = os.path.join(root, f)
        size = os.path.getsize(filepath)
        print(f"{sub_indent}{f}  ({size} bytes)")
    if len(files) > 5:
        print(f"{sub_indent}... ({len(files)} files total)")

In [ ]:
# 预览 cameras.txt
print("=== cameras.txt ===")
with open(os.path.join(OUTPUT_DIR, 'cameras.txt'), 'r') as f:
    print(f.read())

In [ ]:
# 预览 images.txt (前 10 行)
print("=== images.txt (前 10 行) ===")
with open(os.path.join(OUTPUT_DIR, 'images.txt'), 'r') as f:
    for i, line in enumerate(f):
        if i >= 10:
            print("...")
            break
        print(line, end='')

In [ ]:
# 随机展示几张生成的频谱图
spectrum_dir = os.path.join(OUTPUT_DIR, 'spectrum')
all_images = sorted(os.listdir(spectrum_dir))

n_show = min(8, len(all_images))
fig, axes = plt.subplots(2, n_show // 2, figsize=(16, 6))
axes = axes.flatten()

show_indices = np.linspace(0, len(all_images) - 1, n_show, dtype=int)
for ax, idx in zip(axes, show_indices):
    img = imageio.imread(os.path.join(spectrum_dir, all_images[idx]))
    ax.imshow(img)
    ax.set_title(all_images[idx], fontsize=8)
    ax.axis('off')

plt.suptitle('生成的频谱图像样例', fontsize=12)
plt.tight_layout()
plt.show()